> ⏱ ~25 min

# Round 4 — Multi-agent concierge with tools

Rounds 1, 2, and 3 all evaluated a **single** agent — a bare model, then a grounded model, then a grounded model graded by a custom auditor. Round 4 swaps that single agent for the full multi-agent **concierge** you built in `02-build-multi-agent.ipynb`: a coordinator that delegates to specialist flight, hotel, and car agents, each backed by real booking tools over the corporate catalog.

You then rerun the **exact same evaluator panel** as round 3 — built-in graders plus the custom `PolicyAdherenceEvaluator` — on the **exact same** 20-row dataset. The only thing that changes between rounds 3 and 4 is the target: prompt-grounding versus an actual tool-using multi-agent system.

## What you will do

1. Build the multi-agent concierge (flight + hotel + car specialists with booking tools).
2. Wrap it as a synchronous callable that `azure-ai-evaluation.evaluate(...)` can drive.
3. Reuse the same evaluator panel as round 3 (built-ins + `PolicyAdherenceEvaluator`).
4. Save the metrics to `eval-results/round4-multi-agent-*` and compare across all four rounds.

This round closes the loop on the progression: bare model → grounded prompt → grounded prompt with custom judge → multi-agent system with tools, all judged by the same panel against the same dataset.

---

In [ ]:
# Suppress experimental/deprecation warnings to keep the learning output clean.
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os, sys, asyncio
from dotenv import load_dotenv

# Locate this lab folder portably (do not hardcode the repo name; learners may fork+rename).
LAB_DIR = Path.cwd().resolve()
if LAB_DIR.name != 'lab-1-foundry-maf':
    for _p in (LAB_DIR, *LAB_DIR.parents):
        _candidate = _p / 'cohere' / 'lab-1-foundry-maf'
        if _candidate.exists():
            LAB_DIR = _candidate
            break
COHERE_DIR = LAB_DIR.parent
load_dotenv(COHERE_DIR / '.env')
sys.path.insert(0, str(LAB_DIR))

PROJECT_ENDPOINT = os.environ['FOUNDRY_PROJECT_ENDPOINT']
DEPLOYMENT = os.getenv('COMMAND_A_DEPLOYMENT', 'command-a')
DATASET = LAB_DIR / '04-eval-dataset.jsonl'
print('Dataset rows:', sum(1 for _ in DATASET.open(encoding='utf-8')))


## 1. Build the concierge target

`azure-ai-evaluation.evaluate(...)` expects a synchronous callable that takes a single row of fields and returns a dict. The MAF concierge is async, so we wrap it with `asyncio.run` and return the response text in the schema the evaluators expect.

We build the concierge once at module scope so every row reuses the same client connection.


In [ ]:
import threading
from agents import make_chat_client, build_concierge

client = make_chat_client()
concierge = build_concierge(client)

# A single background thread owns one event loop. Both the kernel main
# thread (sanity call) and promptflow's worker threads (evaluate runs) hand
# their coroutines to this loop via run_coroutine_threadsafe, so MAF's
# telemetry ContextVar tokens are always set and reset inside the same task
# context -- and we never touch the kernel's loop, which is what triggered
# the earlier 'Leaving task does not match' warnings under nest_asyncio.
_runner_loop = asyncio.new_event_loop()
_runner_thread = threading.Thread(
    target=_runner_loop.run_forever, name='concierge-loop', daemon=True
)
_runner_thread.start()

async def _invoke_concierge(query: str):
    return await concierge.run(query)

def target(query: str, **kwargs: object) -> dict:
    """Run the MAF concierge for one dataset row and return the response.

    Failures are swallowed and reported in the row instead of raising, so a
    single bad row (for example, a content-filtered response that the
    azure-ai-evaluation SDK cannot parse) does not fail the whole batch and
    block evaluators from running on the other rows.
    """
    try:
        fut = asyncio.run_coroutine_threadsafe(_invoke_concierge(query), _runner_loop)
        response = fut.result()
        text = response.text or ''
        return {'response': text, 'query': query, 'error': ''}
    except Exception as exc:
        return {
            'response': f'[concierge call failed: {type(exc).__name__}: {exc}]',
            'query': query,
            'error': f'{type(exc).__name__}: {exc}',
        }

# Quick sanity call before the long evaluate() run.
print(target('Find an economy flight from SFO to NYC on 2026-07-10 returning 2026-07-13 under $400.')['response'][:400])


## 2. Build the evaluators

Different versions of `azure-ai-evaluation` expose different evaluators. The helper below picks up only the ones that are present in your install, so the cell does not break across SDK versions. Each evaluator receives the same `model_config` so they all judge with Cohere `command-a`.


In [ ]:
from azure.identity import DefaultAzureCredential

# Cohere `command-a` is reachable on the account-level OpenAI-compatible path
# (/openai/v1/chat/completions), NOT on the legacy Azure OpenAI deployment path
# (/openai/deployments/{name}/chat/completions). So the evaluators' judge model
# must be configured as `type: openai` with base_url pointing at /openai/v1.
# We pass an Entra access token as the api_key (valid ~1 hour, well over the
# few minutes an eval run takes).
credential = DefaultAzureCredential()
_judge_token = credential.get_token('https://cognitiveservices.azure.com/.default').token
model_config = {
    'type': 'openai',
    'base_url': os.environ['AZURE_AI_ENDPOINT'].rstrip('/') + '/openai/v1',
    'model': DEPLOYMENT,
    'api_key': _judge_token,
}

def build_evaluators():
    import azure.ai.evaluation as evaluation
    evaluators = {}
    for name in ['RelevanceEvaluator', 'CoherenceEvaluator', 'FluencyEvaluator', 'GroundednessEvaluator']:
        cls = getattr(evaluation, name, None)
        if cls is None:
            continue
        try:
            evaluators[name.replace('Evaluator', '').lower()] = cls(model_config)
        except Exception as exc:
            # Preview versions sometimes change the config schema; skip rather than break the run.
            print(f'Skipping {name}: {exc}')
    # ToolCallAccuracyEvaluator is intentionally skipped: it requires
    # `tool_definitions` and `tool_calls` columns that the local concierge
    # target does not emit. Add it back once those are captured per row.
    for name in ['IntentResolutionEvaluator', 'TaskAdherenceEvaluator']:
        cls = getattr(evaluation, name, None)
        if cls is None:
            continue
        try:
            evaluators[name.replace('Evaluator', '').lower()] = cls(model_config)
        except Exception as exc:
            print(f'Skipping {name}: {exc}')
    return evaluators

# Reuse the exact same panel as round 3: built-ins + custom policy auditor.
# Force-reload the patched evaluator so a kernel restart isn't required after edits.
import sys as _sys
for _k in [k for k in list(_sys.modules) if k.startswith('evaluators')]:
    del _sys.modules[_k]
from evaluators.policy_adherence import PolicyAdherenceEvaluator  # noqa: E402
evaluators = build_evaluators()
evaluators['policy_adherence'] = PolicyAdherenceEvaluator()
print('Active evaluators:', sorted(evaluators))

## 3. Run the evaluation

`evaluate(...)` walks every row in the dataset, calls `target(query=row['query'])`, then hands the result to every evaluator. The aggregate metrics come back in `result.metrics`. Uploading to the Foundry project is optional; when the project coordinates are present the run appears under the Evaluations tab alongside any earlier evaluation runs.


In [ ]:
from datetime import datetime
from azure.ai.evaluation import evaluate
import json as _json
import pandas as pd

# Cap parallel target + evaluator calls so the 100-RPM command-a deployment
# is not bursted past its rate limit. With 6 judge evaluators per row, even
# 4 workers will trigger 429s; 2 keeps the burst comfortably under 100 RPM.
# Set this BEFORE evaluate() builds its batch engine. Use plain assignment
# (not setdefault) so re-running the cell without a kernel restart picks up
# the new value if you ever tune it.
os.environ['PF_WORKER_COUNT'] = '2'

# Hub-less Microsoft Foundry projects are identified by their project endpoint;
# the legacy AzureAIProject(subscription, resource_group, project_name) triple
# resolves to an ML workspace that does not exist for these projects. Passing
# the endpoint string lets evaluate() upload to the right resource. The upload
# itself is best-effort so a portal failure never blocks the local snapshots.
azure_ai_project = PROJECT_ENDPOINT

RESULTS_DIR = LAB_DIR / 'eval-results'
RESULTS_DIR.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
output_path = RESULTS_DIR / f'round4-multi-agent-{timestamp}.json'

def _run_evaluation(project):
    return evaluate(
        data=str(DATASET),
        target=target,
        evaluators=evaluators,
        azure_ai_project=project,
        evaluation_name='round4-multi-agent',
        fail_on_evaluator_errors=False,
        tags={'lab': 'cohere-lab-1-foundry-maf', 'round': 'round4-multi-agent'},
        output_path=str(output_path),
    )

try:
    result = _run_evaluation(azure_ai_project)
    uploaded = True
except Exception as exc:
    print(f'Portal upload failed ({type(exc).__name__}: {exc}); rerunning local-only.')
    result = _run_evaluation(None)
    uploaded = False

metrics = getattr(result, 'metrics', None) or (result.get('metrics') if isinstance(result, dict) else {})
rows = getattr(result, 'rows', None) or (result.get('rows') if isinstance(result, dict) else [])

# evaluate() writes the combined result to output_path. Also save tidy companion
# files so the per-row table and metrics summary are easy to diff across runs.
rows_df = pd.DataFrame(rows)
rows_path = RESULTS_DIR / f'round4-multi-agent-{timestamp}-rows.csv'
metrics_path = RESULTS_DIR / f'round4-multi-agent-{timestamp}-metrics.json'
rows_df.to_csv(rows_path, index=False)
metrics_path.write_text(_json.dumps(metrics, indent=2, sort_keys=True))

print(f'Portal upload: {"yes" if uploaded else "no (local-only)"}')
print(f'Full result : {output_path}')
print(f'Per-row CSV : {rows_path}')
print(f'Metrics JSON: {metrics_path}')
pd.DataFrame(sorted(metrics.items()), columns=['metric', 'value'])

## What you confirmed

- The multi-agent concierge can be wrapped as a sync callable for `azure-ai-evaluation.evaluate`, exactly like the single-agent rounds.
- The same evaluator panel — built-ins plus your custom `PolicyAdherenceEvaluator` — works against every round, so all four runs are directly comparable.
- You now have four metrics files in `eval-results/` (`round1-baseline-*`, `round2-grounded-*`, `round3-policy-evaluator-*`, `round4-multi-agent-*`) that tell the full progression story: bare model → grounded prompt → custom judge → multi-agent system with tools.

Next: **`05-local-redteam.ipynb`** runs a safety scan against the same multi-agent concierge.